# 05 · Bandits, MDPs, and value iteration

**Goal:** meet the exploration/exploitation trade-off in its simplest form (bandits), then the full machinery of Markov decision processes and the Bellman equations. You will *verify* that value iteration is a contraction.

## Part A: the $k$-armed bandit

Arm $a$ pays a random reward with unknown mean $q_*(a)$. Keep a sample-average estimate with the incremental update

$$Q_{n+1}=Q_n+\frac1{n+1}\,(R_{n+1}-Q_n),$$

and with probability $\varepsilon$ pick a random arm ($\varepsilon$-greedy).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def run_bandit(eps, runs=500, steps=1000, k=10, seed=0):
    rng = np.random.default_rng(seed)
    qstar = rng.normal(size=(runs, k))
    Q = np.zeros((runs, k)); Ncount = np.zeros((runs, k))
    idx = np.arange(runs)
    avg = np.zeros(steps)
    for t in range(steps):
        greedy = np.argmax(Q + 1e-9 * rng.random((runs, k)), axis=1)   # random tie-breaking
        explore = rng.random(runs) < eps
        a = np.where(explore, rng.integers(k, size=runs), greedy)
        r = qstar[idx, a] + rng.normal(size=runs)
        Ncount[idx, a] += 1
        Q[idx, a] += (r - Q[idx, a]) / Ncount[idx, a]
        avg[t] = r.mean()
    return avg

curves = {eps: run_bandit(eps) for eps in (0.0, 0.01, 0.1)}
for eps, c in curves.items():
    plt.plot(c, label=f"eps={eps}")
    print(f"eps={eps:<5} mean reward over last 200 steps: {c[-200:].mean():.3f}")
plt.xlabel("step"); plt.ylabel("average reward"); plt.legend(); plt.show()
assert curves[0.1][-200:].mean() > curves[0.0][-200:].mean()

## Part B: Markov decision processes

An MDP is $(\mathcal S,\mathcal A,P,R,\gamma)$. A policy $\pi$ has value $V^\pi(s)=\mathbb E_\pi[\sum_t\gamma^tR_t\mid s_0=s]$, which satisfies the **Bellman equation**

$$V^\pi=R_\pi+\gamma P_\pi V^\pi \;\Longleftrightarrow\; (I-\gamma P_\pi)V^\pi=R_\pi ,$$

a *linear system* (so policy evaluation is linear algebra). The optimal value satisfies the **Bellman optimality equation**

$$V^*(s)=\max_a\Big[R(s,a)+\gamma\sum_{s'}P(s'|s,a)V^*(s')\Big].$$

The operator $T V=\max_a[R+\gamma PV]$ is a $\gamma$-**contraction** in the sup norm: $\|TV-TW\|_\infty\le\gamma\|V-W\|_\infty$. Banach's fixed-point theorem then gives a unique $V^*$ and convergence of $V\leftarrow TV$ (value iteration).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 4x4 gridworld, actions: 0=up 1=right 2=down 3=left, terminals in two opposite corners,
# reward -1 per step (so the optimal policy takes the shortest path to a corner).
N = 4
S = N * N
A = 4
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]
TERMINALS = {0, S - 1}

P = np.zeros((S, A, S))        # P[s, a, s'] transition probabilities
R = np.full((S, A), -1.0)      # R[s, a] expected reward
for s in range(S):
    if s in TERMINALS:
        P[s, :, s] = 1.0
        R[s, :] = 0.0
        continue
    r, c = divmod(s, N)
    for a, (dr, dc) in enumerate(MOVES):
        r2, c2 = r + dr, c + dc
        if not (0 <= r2 < N and 0 <= c2 < N):
            r2, c2 = r, c                      # bumping into a wall = stay put
        P[s, a, r2 * N + c2] = 1.0
assert np.allclose(P.sum(axis=2), 1.0)         # every row is a probability distribution

In [ ]:
gamma = 0.9

def bellman_opt(V):
    Q = R + gamma * P @ V          # shape (S, A)
    return Q.max(axis=1), Q

# run value iteration to (near) machine precision to get V*
V = np.zeros(S)
for _ in range(2000):
    V_new, _ = bellman_opt(V)
    if np.max(np.abs(V_new - V)) < 1e-14:
        break
    V = V_new
V_star = V
print(V_star.reshape(N, N).round(3))

# verify the contraction: error shrinks by at least a factor gamma every sweep
V = np.zeros(S)
errs = [np.max(np.abs(V - V_star))]
for _ in range(40):
    V, _ = bellman_opt(V)
    errs.append(np.max(np.abs(V - V_star)))
ratios = [errs[i + 1] / errs[i] for i in range(len(errs) - 1) if errs[i] > 1e-9]
print("max error ratio between sweeps:", max(ratios).round(4), "<= gamma =", gamma)
assert max(ratios) <= gamma + 1e-9

## Extract the greedy policy and evaluate it with a linear solve

In [ ]:
_, Q_star = bellman_opt(V_star)
pi = Q_star.argmax(axis=1)

arrows = np.array(["^", ">", "v", "<"])
grid = np.array([arrows[pi[s]] if s not in TERMINALS else "." for s in range(S)])
print(grid.reshape(N, N))

# policy evaluation: solve (I - gamma P_pi) V = R_pi
P_pi = P[np.arange(S), pi]         # (S, S)
R_pi = R[np.arange(S), pi]
V_pi = np.linalg.solve(np.eye(S) - gamma * P_pi, R_pi)
assert np.allclose(V_pi, V_star, atol=1e-9)
print("policy evaluation by linear solve matches V*")

## Exercises

1. Implement **policy iteration** (evaluate by linear solve, improve greedily) and count how many iterations it needs compared to value iteration.
2. Make the world stochastic: with probability 0.2 the agent slips to a random neighbouring action. Rebuild `P` and re-solve. Does the policy change?
3. Show numerically that the spectral radius of $\gamma P_\pi$ is $\le\gamma$, so $I-\gamma P_\pi$ is always invertible.